In [28]:
import os
import polars as pl
from tqdm.notebook import tqdm
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")

polars.config.Config

| Statistic | Delay (hours) |
|---|---:|
| Count | 66,106,991 |
| Null count | 696,048 |
| Mean | 1.479 |
| Standard deviation | 6.791 |
| Minimum | -6263.000 |
| 25% | 0.000 |
| 50% | 1.000 |
| 75% | 1.000 |
| Maximum | 8760.000 |

In [41]:
from collections import defaultdict
import os
import polars as pl
from tqdm.notebook import tqdm

data_path = '/scratch/sas10092/ehr-foundation/data/meds_normalized/data/train/'
downstream_idx = pl.read_parquet('/scratch/sas10092/ehr-foundation/resources/downstream_idx.parquet')


audit = {
    "within24": {
        "total": 0,
        "with_diagnosis": 0,
        "diagnosis_events": 0,
    },
    "within48": {
        "total": 0,
        "with_diagnosis": 0,
        "diagnosis_events": 0,
    },
    "within_stay": {
        "total": 0,
        "with_diagnosis": 0,
        "diagnosis_events": 0,
    },
}


for i in tqdm(range(len(downstream_idx))):

    stay = downstream_idx[i]

    subject_id = stay['subject_id'][0]
    shard = stay['shard'][0]

    q_start_24 = stay['w24_start_1024'][0]
    q_end_24 = stay['w24_end_1024'][0]

    q_start_48 = stay['w48_start_1024'][0]
    q_end_48 = stay['w48_end_1024'][0]

    q_start_st = stay['wStay_start_1024'][0]
    q_end_st = stay['wStay_end_1024'][0]


    shard_df = pl.read_parquet(
        os.path.join(data_path, shard)
    )

    timeline = shard_df.filter(
        pl.col('subject_id') == subject_id
    )


    queries = {
        "within24": timeline[q_start_24:q_end_24],
        "within48": timeline[q_start_48:q_end_48],
        "within_stay": timeline[q_start_st:q_end_st],
    }


    for name, query in queries.items():

        diagnosis_events = query.filter(
            pl.col('code').str.starts_with('DIAGNOSIS')
        )

        n_diag = diagnosis_events.height

        audit[name]["total"] += 1

        if n_diag > 0:
            audit[name]["with_diagnosis"] += 1

        audit[name]["diagnosis_events"] += n_diag



summary = []

for window, values in audit.items():

    summary.append({
        "window": window,
        "total_queries": values["total"],
        "queries_with_diagnosis": values["with_diagnosis"],
        "diagnosis_event_count": values["diagnosis_events"],
        "diagnosis_presence_rate": 
            values["with_diagnosis"] / values["total"]
    })


audit_summary = pl.DataFrame(summary)

audit_summary.write_parquet('audit_summary.parquet')

  0%|          | 0/61175 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
# audit

In [12]:
# timeline.filter(pl.col('code').str.starts_with('DIAGNOSIS'))

In [13]:
# downstream_idx.filter(pl.col('subject_id') == 10024043)

In [15]:
leakage_candidates = []
for i in tqdm(range(len(downstream_idx))):

    stay = downstream_idx[i]

    subject_id = stay["subject_id"][0]
    stay_id = stay["icustay_id"][0]
    shard = stay["shard"][0]

    shard_df = pl.read_parquet(
        os.path.join(data_path, shard)
    )

    timeline = shard_df.filter(
        pl.col("subject_id") == subject_id
    )

    windows = {
        "within24": (
            stay["w24_start_1024"][0],
            stay["w24_end_1024"][0],
        ),
        "within48": (
            stay["w48_start_1024"][0],
            stay["w48_end_1024"][0],
        ),
        "within_stay": (
            stay["wStay_start_1024"][0],
            stay["wStay_end_1024"][0],
        ),
    }
    
    for name, (start, end) in windows.items():

        query = timeline[start:end]

        diag = query.filter(
            pl.col("code").str.starts_with("DIAGNOSIS")
        )
        
        if diag.height > 0:
            
            leakage_candidates.append({
                "icustay_id": stay_id,
                "subject_id": subject_id,
                "window": name,
                "query_start_idx": start,
                "query_end_idx": end,
                "diagnosis_codes": diag["code"].to_list(),
                "diagnosis_times": diag["time"].to_list(),
            })


  0%|          | 0/61175 [00:00<?, ?it/s]

31342860
32895909
33060379
38688395
34510292
39558308
36575425
36081309
39492446
38860085
33958400
31559995
37521741
34728076
38922429
31350931
38989978
36594826
36827644
35736966
38860905
37801913
37801913
31097260
30321781
37831823
30950523
39911054
39911054
30273492
34706547
30546860
30492740
39789769
37804132
30137539
36091557
37915515
36161672
34619266
32381432
36324659
39242254
30354498
32040322
38552607
35141168
37898603
39485910
34969374
31969135
35896096
34692268
39953985
31145488
37159783
39177547
33349381


KeyboardInterrupt: 

In [29]:
for i in range(len(leakage_candidates)):
    if leakage_candidates[i]['window'] == 'within24':
        print(i)
    elif leakage_candidates[i]['window'] == 'within48':
        print(i)

13
17
19
24
31
36
40
42
46
48
52
56
58
65
68
70
72
77
87
92
106
108
109
111
113
115
121
124
125
135
137
141
144
147
152
155
157
159
161
165
168
171
173
175
180
183
186
188
198
200
204
210
215
218
221
223
226
228


In [33]:
i = 108

In [34]:
subject_id = leakage_candidates[i]['subject_id']
print(subject_id)
icustay_id = leakage_candidates[i]['icustay_id']
print(icustay_id)
window = leakage_candidates[i]['window']
print(window)
 
query_start_idx = leakage_candidates[i]['query_start_idx']
print(query_start_idx)
query_end_idx = leakage_candidates[i]['query_end_idx']
print(query_end_idx)

stay = downstream_idx.filter(pl.col('icustay_id') == icustay_id)

print('\n------------\n')


subject_id = stay['subject_id'][0]
print(subject_id)
shard = stay['shard'][0]
print(shard)

if window == 'within24':
    q_start_24 = stay['w24_start_1024'][0]
    print(q_start_24)
    q_end_24 = stay['w24_end_1024'][0]
    print(q_end_24)

if window == 'within48':
    q_start_48 = stay['w48_start_1024'][0]
    print(q_start_48)
    q_end_48 = stay['w48_end_1024'][0]
    print(q_end_48)

if window == 'within_stay':
    q_start_st = stay['wStay_start_1024'][0]
    print(q_start_st)
    q_end_st = stay['wStay_end_1024'][0]
    print(q_end_st)

shard_df = pl.read_parquet(
    os.path.join(data_path, shard)
)

timeline = shard_df.filter(
    pl.col('subject_id') == subject_id
)

10067195
37801913
within24
268
1291

------------

10067195
261.parquet
268
1291


In [40]:
timeline[query_start_idx:query_end_idx]#.filter(pl.col('code').str.starts_with('DIAG'))

subject_id,seq_id,out_id,er_id,hadm_id,icustay_id,disch_id,time,code,numeric_value,text_value,itemid,died_in_hosp,icu_los,admission_type,admission_location,discharge_location,diag_version,diag_icd_code,diag_seq_num,drg_severity,drg_mortality,drg_type,drg_code,priority,specimen_id,lab_lower_limit,lab_upper_limit,lab_flag,lab_unit,lab_itemid,gender,route,frequency,doses_per_24_hrs,medication,proc_seq_num,proc_version,proc_icd_code,micro_specimen_id,micro_org_name,micro_test_name,micro_spec_type_desc,micro_test_itemid,icu_care_unit,category,label,abbreviation,rate,unit,amount,amountuom,ordercategorydescription,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,table,race,code_type,icd9_to_icd10_d,icd9_to_icd10_p,clean_medication,lab_label,lab_fluid,lab_category,lab_description,lab_frequency,time_diff,numeric_value/is_inlier
i64,f64,f64,f64,f64,f64,f64,datetime[μs],str,f32,str,f64,f64,f64,str,str,str,f64,str,f64,f64,f64,str,f64,str,f64,f64,f64,str,str,f64,str,str,str,f64,str,f64,f64,str,f64,str,str,str,f64,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,bool
10067195,21564201,null,null,21564201,37801913,null,2181-08-27 09:14:00,"""ICU-CHART//225110//Recreationa…",null,null,225110,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Adm History/FHPA""","""Recreational drug use""","""Recreational drug use""",null,null,null,null,null,null,null,null,"""icu/chartevents""",null,"""ICU-CHART""",null,null,"""UNK""",null,null,null,null,null,0,true
10067195,21564201,null,null,21564201,37801913,null,2181-08-27 09:14:00,"""ICU-CHART//225118//Difficulty …",-0.37331923842430115,null,225118,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Adm History/FHPA""","""Difficulty swallowing""","""Difficulty swallowing""",null,null,null,null,null,null,null,null,"""icu/chartevents""",null,"""ICU-CHART""",null,null,"""UNK""",null,null,null,null,null,0,true
10067195,21564201,null,null,21564201,37801913,null,2181-08-27 09:14:00,"""ICU-CHART//225120//Appetite""",null,"""Poor""",225120,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Adm History/FHPA""","""Appetite""","""Appetite""",null,null,null,null,null,null,null,null,"""icu/chartevents""",null,"""ICU-CHART""",null,null,"""UNK""",null,null,null,null,null,0,null
10067195,21564201,null,null,21564201,37801913,null,2181-08-27 09:14:00,"""ICU-CHART//225122//Special die…",1.6800487041473389,null,225122,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Adm History/FHPA""","""Special diet""","""Special diet""",null,null,null,null,null,null,null,null,"""icu/chartevents""",null,"""ICU-CHART""",null,null,"""UNK""",null,null,null,null,null,0,true
10067195,21564201,null,null,21564201,37801913,null,2181-08-27 09:14:00,"""ICU-CHART//225124//Unintention…",2.638476610183716,null,225124,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Adm History/FHPA""","""Unintentional weight loss >10 …","""Unintentional weight loss >10 …",null,null,null,null,null,null,null,null,"""icu/chartevents""",null,"""ICU-CHART""",null,null,"""UNK""",null,null,null,null,null,0,true
10067195,21564201,null,null,21564201,37801913,null,2181-08-27 09:14:00,"""ICU-CHART//225126//Dialysis pa…",null,null,225126,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Adm History/FHPA""","""Dialysis

In [39]:
stay

subject_id,hadm_id,hosp_admission_time,hosp_discharge_time,icustay_id,icu_admission_time,icu_discharge_time,in_hosp_mort_time,out_mortality_time,n_events_hosp,n_events_icu,shard,hosp_los,hosp_los_hours,hosp_los_days,icu_los,icu_los_hours,icu_los_days,mort_24hr_offset,mort_48hr_offset,y_mort,y_mort_1yr,y_mort_9mo,y_mort_6mo,y_mort_3mo,y_los_7,y_los_15,y_los_30,y_icu_readmit,y_icu_readmit_7,y_icu_readmit_15,y_icu_readmit_30,split,w24_min,w24_max,w48_min,w48_max,wStay_min,wStay_max,w24_start_512,w24_end_512,w24_start_1024,w24_end_1024,w24_start_1536,w24_end_1536,w24_start_2048,w24_end_2048,w48_start_512,w48_end_512,w48_start_1024,w48_end_1024,w48_start_1536,w48_end_1536,w48_start_2048,w48_end_2048,wStay_start_512,wStay_end_512,wStay_start_1024,wStay_end_1024,wStay_start_1536,wStay_end_1536,wStay_start_2048,wStay_end_2048
i64,i64,datetime[μs],datetime[μs],i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],u32,u32,str,duration[μs],f64,f64,duration[μs],f64,f64,datetime[μs],datetime[μs],i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
10067195,21564201,2181-08-27 06:48:00,2181-08-28 00:00:00,37801913,2181-08-27 08:26:00,2181-08-29 00:01:24,null,2181-09-24 00:00:00,1838,1751,"""261.parquet""",17h 12m,17.2,0.7166666666666667,1d 15h 35m 24s,39.59,1.6495833333333332,2181-08-28 08:26:00,2181-08-29 08:26:00,0,1,1,1,1,0,0,0,0,0,0,0,"""train""",100,1291,100,1930,100,1930,780,1291,268,1291,100,1291,100,1291,1419,1930,907,1930,395,1930,100,1930,1419,1930,907,1930,395,1930,100,1930


In [73]:
audit_df = audit_df.with_columns(
    pl.col("diagnosis_times").cast(pl.Datetime)
)

In [115]:
# list(audit_df['subject_id'].unique())

In [102]:
# downstream_idx.filter(pl.col('icustay_id') == 38392119)

In [116]:
# stay = downstream_idx[0]

# subject_id = stay['subject_id'][0]
# shard = stay['shard'][0]

# q_start_24 = stay['w24_start_1024'][0]
# q_end_24 = stay['w24_end_1024'][0]

# q_start_48 = stay['w48_start_1024'][0]
# q_end_48 = stay['w48_end_1024'][0]

# q_start_st = stay['wStay_start_1024'][0]
# q_end_st = stay['wStay_end_1024'][0]

In [ ]:
import os
import polars as pl
from tqdm.notebook import tqdm
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")

In [11]:
data_path = '/scratch/sas10092/ehr-foundation/data/meds_normalized/data/train/'
downstream_idx = pl.read_parquet('/scratch/sas10092/ehr-foundation/resources/downstream_idx.parquet')

delay_threshold_hours = 1.5

audit = {
    "within24": {
        "samples": 0,
        "samples_with_labs": 0,
        "samples_with_potential_leakage": 0,
        "total_labs": 0,
        "potential_leaked_labs": 0,
    },
    "within48": {
        "samples": 0,
        "samples_with_labs": 0,
        "samples_with_potential_leakage": 0,
        "total_labs": 0,
        "potential_leaked_labs": 0,
    },
    "within_stay": {
        "samples": 0,
        "samples_with_labs": 0,
        "samples_with_potential_leakage": 0,
        "total_labs": 0,
        "potential_leaked_labs": 0,
    },
}


for i in tqdm(range(len(downstream_idx))):

    stay = downstream_idx[i]

    subject_id = stay['subject_id'][0]
    shard = stay['shard'][0]

    q_start_24 = stay['w24_start_1024'][0]
    q_end_24 = stay['w24_end_1024'][0]

    q_start_48 = stay['w48_start_1024'][0]
    q_end_48 = stay['w48_end_1024'][0]

    q_start_st = stay['wStay_start_1024'][0]
    q_end_st = stay['wStay_end_1024'][0]


    shard_df = pl.read_parquet(
        os.path.join(data_path, shard)
    )

    timeline = shard_df.filter(
        pl.col('subject_id') == subject_id
    )


    windows = {
        "within24": (q_start_24, q_end_24),
        "within48": (q_start_48, q_end_48),
        "within_stay": (q_start_st, q_end_st),
    }


    for name, (start, end) in windows.items():

        audit[name]["samples"] += 1

        query = timeline[start:end]

        labs = query.filter(
            pl.col("code").str.starts_with("LAB")
        )

        if labs.height == 0:
            continue

        audit[name]["samples_with_labs"] += 1
        audit[name]["total_labs"] += labs.height


        # last timestamp in query = prediction cutoff
        cutoff = query["time"].max()


        # labs occurring close to the cutoff
        potential = labs.filter(
            (
                cutoff - pl.col("time")
            ).dt.total_hours()
            <= delay_threshold_hours
        )


        n_potential = potential.height

        audit[name]["potential_leaked_labs"] += n_potential

        if n_potential > 0:
            audit[name]["samples_with_potential_leakage"] += 1



summary = []

for window, values in audit.items():

    summary.append({
        "window": window,
        "samples": values["samples"],
        "samples_with_labs": values["samples_with_labs"],
        "samples_with_potential_leakage": values["samples_with_potential_leakage"],
        "total_labs": values["total_labs"],
        "potential_leaked_labs": values["potential_leaked_labs"],
        "sample_leakage_rate": (
            values["samples_with_potential_leakage"]
            / values["samples"]
        ),
        "lab_leakage_rate": (
            values["potential_leaked_labs"]
            / values["total_labs"]
            if values["total_labs"] > 0 else 0
        ),
    })


audit_summary = pl.DataFrame(summary)

lab_audit_summary.write_parquet('lab_audit.parquet')

 16%|█▌        | 9594/61175 [26:34<2:22:52,  6.02it/s] 


KeyboardInterrupt: 

In [12]:
summary = []

for window, values in audit.items():

    summary.append({
        "window": window,
        "samples": values["samples"],
        "samples_with_labs": values["samples_with_labs"],
        "samples_with_potential_leakage": values["samples_with_potential_leakage"],
        "total_labs": values["total_labs"],
        "potential_leaked_labs": values["potential_leaked_labs"],
        "sample_leakage_rate": (
            values["samples_with_potential_leakage"]
            / values["samples"]
        ),
        "lab_leakage_rate": (
            values["potential_leaked_labs"]
            / values["total_labs"]
            if values["total_labs"] > 0 else 0
        ),
    })


audit_summary = pl.DataFrame(summary)

In [13]:
audit_summary

window,samples,samples_with_labs,samples_with_potential_leakage,total_labs,potential_leaked_labs,sample_leakage_rate,lab_leakage_rate
str,i64,i64,i64,i64,i64,f64,f64
"""within24""",9594,9269,2734,465581,41943,0.28497,0.090087
"""within48""",9594,9161,2136,391208,29505,0.222639,0.07542
"""within_stay""",9594,8969,1238,346894,13310,0.129039,0.038369


In [15]:
pl.read_parquet('../audit_summary.parquet')

window,total_queries,queries_with_diagnosis,diagnosis_event_count,diagnosis_presence_rate
str,i64,i64,i64,f64
"""within24""",61175,186,2976,0.00304
"""within48""",61175,3216,44476,0.05257
"""within_stay""",61175,11128,208692,0.181904


In [98]:
rgp = pl.read_csv('./significance/ehrragp_y_los_7.csv')
rfrmr = pl.read_csv('./significance/vanilla_y_los_7.csv')

In [99]:
rgp.join(rfrmr, on=['icustay_id','label'], how='inner')

icustay_id,label,prediction,prediction_right
i64,f64,f64,f64
31205490,0.0,0.006104,0.006104
35044219,0.0,0.102539,0.102539
32773003,1.0,0.660156,0.5078125
37919158,0.0,0.318359,0.214844
38292466,0.0,0.425781,0.322266
…,…,…,…
30444664,0.0,0.042725,0.022339
36469520,0.0,0.005737,0.005737
30988867,0.0,0.84375,0.722656


In [100]:
threshold = 0.5

df = rgp.join(
    rfrmr,
    on=['icustay_id','label'],
    how='inner'
)

df = df.with_columns([
    (pl.col("prediction") >= threshold).cast(pl.Int8).alias("rgp_pred_class"),
    (pl.col("prediction_right") >= threshold).cast(pl.Int8).alias("baseline_pred_class"),
])

df = df.with_columns([
    (pl.col("rgp_pred_class") == pl.col("label")).alias("rgp_correct"),
    (pl.col("baseline_pred_class") == pl.col("label")).alias("baseline_correct"),
])

In [101]:
df.filter(
    (~pl.col("baseline_correct")) &
    (pl.col("rgp_correct"))
)

icustay_id,label,prediction,prediction_right,rgp_pred_class,baseline_pred_class,rgp_correct,baseline_correct
i64,f64,f64,f64,i8,i8,bool,bool
32743332,1.0,0.671875,0.488281,1,0,true,false
39699336,1.0,0.609375,0.484375,1,0,true,false
32631424,0.0,0.490234,0.527344,0,1,true,false
33674470,1.0,0.550781,0.451172,1,0,true,false
34241430,1.0,0.605469,0.470703,1,0,true,false
…,…,…,…,…,…,…,…
35278428,1.0,0.519531,0.466797,1,0,true,false
30357604,1.0,0.515625,0.3828125,1,0,true,false
31991205,1.0,0.550781,0.490234,1,0,true,false


In [102]:
df.filter(
    (pl.col("baseline_correct")) &
    (~pl.col("rgp_correct"))
)

icustay_id,label,prediction,prediction_right,rgp_pred_class,baseline_pred_class,rgp_correct,baseline_correct
i64,f64,f64,f64,i8,i8,bool,bool
38740124,0.0,0.527344,0.486328,1,0,false,true
34146568,1.0,0.353516,0.546875,0,1,false,true
32751084,0.0,0.636719,0.400391,1,0,false,true
38603673,0.0,0.65625,0.46875,1,0,false,true
34996407,0.0,0.636719,0.2734375,1,0,false,true
…,…,…,…,…,…,…,…
35298179,0.0,0.636719,0.4765625,1,0,false,true
30852793,0.0,0.535156,0.470703,1,0,false,true
30316992,0.0,0.636719,0.451172,1,0,false,true


In [103]:
df.filter(
    pl.col("baseline_correct") &
    pl.col("rgp_correct")
)

icustay_id,label,prediction,prediction_right,rgp_pred_class,baseline_pred_class,rgp_correct,baseline_correct
i64,f64,f64,f64,i8,i8,bool,bool
31205490,0.0,0.006104,0.006104,0,0,true,true
35044219,0.0,0.102539,0.102539,0,0,true,true
32773003,1.0,0.660156,0.5078125,1,1,true,true
37919158,0.0,0.318359,0.214844,0,0,true,true
38292466,0.0,0.425781,0.322266,0,0,true,true
…,…,…,…,…,…,…,…
34817232,0.0,0.017944,0.012451,0,0,true,true
30444664,0.0,0.042725,0.022339,0,0,true,true
36469520,0.0,0.005737,0.005737,0,0,true,true


In [104]:
df.filter(
    (~pl.col("baseline_correct")) &
    (~pl.col("rgp_correct"))
)

icustay_id,label,prediction,prediction_right,rgp_pred_class,baseline_pred_class,rgp_correct,baseline_correct
i64,f64,f64,f64,i8,i8,bool,bool
39801884,0.0,0.816406,0.691406,1,1,false,false
31950308,1.0,0.194336,0.161133,0,0,false,false
37509585,0.0,0.890625,0.894531,1,1,false,false
37086676,1.0,0.3671875,0.15918,0,0,false,false
39061571,0.0,0.890625,0.875,1,1,false,false
…,…,…,…,…,…,…,…
34132866,1.0,0.042725,0.014038,0,0,false,false
37049736,1.0,0.021606,0.021606,0,0,false,false
37029190,0.0,0.6953125,0.675781,1,1,false,false


In [105]:
df.select([
    pl.col("rgp_correct").sum().alias("EHR-RAGp correct"),
    pl.col("baseline_correct").sum().alias("Baseline correct"),
    ((~pl.col("baseline_correct")) & pl.col("rgp_correct")).sum().alias("Fixed by retrieval"),
    (pl.col("baseline_correct") & (~pl.col("rgp_correct"))).sum().alias("Broken by retrieval"),
])

EHR-RAGp correct,Baseline correct,Fixed by retrieval,Broken by retrieval
u32,u32,u32,u32
11184,11168,122,106


In [1]:
import polars as pl

In [63]:
can = pl.read_parquet('../candidates.parquet')

In [64]:
ragp = pl.read_csv('./significance/ehrragp_y_mort_1yr.csv')
# bsln = pl.read_csv('./significance/roformer_y_icu_readmit_30.csv')

In [65]:
# within24
# within48
# within_stay
ids = list(can.filter(pl.col('window') == 'within_stay')['icustay_id'].unique())

In [66]:
len(ids)

11128

In [67]:
df = ragp.filter(pl.col('icustay_id').is_in(ids) == False)

In [68]:
import torch
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision


y_true = torch.tensor(df["label"].to_numpy()).long()
y_score = torch.tensor(df["prediction"].to_numpy()).float()


auroc_metric = BinaryAUROC()
auprc_metric = BinaryAveragePrecision()


auroc = auroc_metric(y_score, y_true)
auprc = auprc_metric(y_score, y_true)


print(f"AUROC: {auroc.item():.3f}")
print(f"AUPRC: {auprc.item():.3f}")

AUROC: 0.795
AUPRC: 0.354


In [69]:
can['window'].unique()

window
str
"""within24"""
"""within48"""
"""within_stay"""


In [ ]:
AUROC: 0.795
AUPRC: 0.354